In [30]:
from pathlib import Path

import numpy as np
import pyvista as pv

from ulac.common import mesh_utils

In [31]:
patient_id = "07"
input_mesh_file = Path(f"../data/patient_{patient_id}/mesh_endo_raw.vtu")
output_mesh_file = Path(f"../data/patient_{patient_id}/mesh_with_fibers_tags.vtu")
decimation_factor = 0.75
num_smoothing_iterations = 100
extraction_threshold = 0.75
dimension_scaling_factor = 1e-3
legacy_tags = {"MV": 11, "LAA": 13, "LIPV": 21, "LSPV": 23, "RIPV": 25, "RSPV": 27}
new_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

In [32]:
input_mesh = pv.read(input_mesh_file)
triangular_mesh = mesh_utils.convert_unstructured_to_polydata_mesh(input_mesh)
del triangular_mesh.point_data["alpha"]
del triangular_mesh.point_data["beta"]
for key, legacy_value in legacy_tags.items():
    anatomical_region = np.where(triangular_mesh.cell_data["anatomical_tags"] == legacy_value)[0]
    triangular_mesh.cell_data["anatomical_tags"][anatomical_region] = new_tags[key]

triangular_mesh = mesh_utils.preprocess_mesh(
    triangular_mesh,
    tag_values=new_tags.values(),
    decimation_factor=decimation_factor,
    num_smoothing_iters=num_smoothing_iterations,
    extraction_threshold=extraction_threshold,
    dimension_scaling_factor=dimension_scaling_factor,
)
output_mesh_file.parent.mkdir(parents=True, exist_ok=True)
pv.save_meshio(output_mesh_file, triangular_mesh)

In [33]:
plotter = pv.Plotter(window_size=[900, 900])
plotter.add_mesh(
    triangular_mesh,
    scalars="anatomical_tags",
    cmap="tab20",
    show_edges=True,
    edge_color="lightgray",
    edge_opacity=0.3,
    show_scalar_bar=False,
)
plotter.show()

Widget(value='<iframe src="http://localhost:39505/index.html?ui=P_0x7f46e7cba850_7&reconnect=auto" class="pyvi…